# Writing an eval

Before training, you need a baseline: how well does the untrained model already perform on your task?
An eval answers that question and doubles as a sanity check for your reward function — if the base model scores 0% or 100%, something is off.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
from modal_training_gym import (
    DeploymentConfig,
    EvalConfig,
    EvalRowResult,
    HuggingFaceDataset,
    Qwen3_4B,
)

## Serve the base model

First, deploy the model just like in the previous tutorial.

In [ ]:
base_model = Qwen3_4B()
base_model_deployment = DeploymentConfig(
    model=base_model,
).serve()
print(f"Base model deployed to {base_model_deployment.url}")

## Define the eval dataset

We'll use [`Onlydrinkwater/int_division`](https://huggingface.co/datasets/Onlydrinkwater/int_division) — a HuggingFace dataset of integer division problems. Each row has a `question` (the problem) and a `label` (the correct answer). Subclass `HuggingFaceDataset` to point at it.

In [ ]:
class IntDivisionDataset(HuggingFaceDataset):
    hf_repo = "Onlydrinkwater/int_division"
    input_column = "question"
    output_column = "label"
    output_format = "jsonl"
    apply_chat_template = True
    system_prompt = (
        "You are a math problem solver. Solve the given math problem."
    )
eval_dataset = IntDivisionDataset(n_rows=20)

Let's take a look at the eval set.

In [ ]:
df = eval_dataset.to_pandas()
print(f"Number of rows in eval set: {len(df)}")
df.head(5)

## Define a scoring function

The scoring function receives each dataset row and the model's response, then returns an `EvalRowResult` with a score. Here we do a simple exact-match: score 1.0 if the response equals the label, 0.0 otherwise.

In [ ]:
def eval_response_fn(_example: dict, response: str) -> EvalRowResult:
    if _example["label"] == response:
        return EvalRowResult(score=1.0, response=response)
    else:
        return EvalRowResult(score=0.0, response=response)

eval_config = EvalConfig(
    dataset=eval_dataset,
    eval_response_fn=eval_response_fn,
)

In [ ]:
print("——— Running base model evaluation... ———")
base_eval = eval_config.evaluate(base_model_deployment, debug=True)
print(f"Average score: {base_eval.mean:.1f}")
print("——— Base model evaluation complete ———")

## Debugging a zero score

A score of 0.0 means none of the responses matched the labels exactly. Let's inspect a response to understand why.

In [ ]:
print(base_eval.rows[0])

## Fixing the eval

Two problems:
1. **Thinking mode is on** — the model emits internal reasoning before the answer, so the response is never just a number.
2. **No format instruction** — even without thinking, the model wraps the answer in natural language.

Training on a reward function that always returns 0 teaches the model nothing. Let's fix all three:
- Disable thinking via `generate_kwargs` and cap `max_tokens` to keep responses short.
- Embed the format instruction in `prompt_template` so it reaches the model at inference time. (Note: `system_prompt` only affects training data preparation — it isn't sent during `generate()` calls.)
- Extract the first integer from the response with a regex instead of requiring an exact string match.

In [ ]:
import re

class FixedIntDivisionDataset(HuggingFaceDataset):
    hf_repo = "Onlydrinkwater/int_division"
    input_column = "question"
    output_column = "label"
    output_format = "jsonl"
    apply_chat_template = True
    prompt_template = (
        "Solve this math problem. Reply with ONLY the integer answer, nothing else.\n\n{input}"
    )

def new_eval_response_fn(_example: dict, response: str) -> EvalRowResult:
    match = re.search(r"-?\d+", response)
    extracted = match.group() if match else ""
    if _example["label"] == extracted:
        return EvalRowResult(score=1.0, response=response)
    else:
        return EvalRowResult(score=0.0, response=response)

new_eval_config = EvalConfig(
    dataset=FixedIntDivisionDataset(),
    eval_response_fn=new_eval_response_fn,
    generate_kwargs={
        "chat_template_kwargs": {"enable_thinking": False},
        "max_tokens": 20,
    },
)

In [ ]:
print("——— Running fixed eval... ———")
fixed_eval = new_eval_config.evaluate(base_model_deployment, debug=True)
print(f"Average fixed eval score: {fixed_eval.mean:.1f}")
print("——— Fixed eval complete ———")